In [5]:
import numpy as np
import pandas as pd
import sys
import torch
import os
import pickle
import json
from collections import defaultdict
sys.path.append("../../../src/experiment_util") 
import experiment_utils
import experiment_constants
sys.path.append("../../../src/irl") 
import data_corruption_utils
import irl_utils
# import pytorch_irl

In [ ]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]

In [7]:
df_w_generated_traj_troll = pd.read_pickle("../../../data/df_w_generated_traj_troll_processed_w_gail_opt.pkl") #pd.read_pickle(dataframes_path + "/df_w_generated_traj_troll_processed_w_gail_opt.pkl")
df_w_generated_traj_organic = pd.read_pickle("../../../data/df_w_generated_traj_organic_processed_w_gail_opt.pkl") #pd.read_pickle(dataframes_path + "/df_w_generated_traj_organic_processed_w_gail_opt.pkl")


In [21]:
set(df_w_generated_traj_troll.columns) - set(z.columns)

{'embed_generated',
 'policy',
 'policy_nt',
 'tp_generated',
 'tp_nt',
 'traj_counts_generated',
 'traj_generated',
 'traj_generated_reshaped',
 'user_nt'}

In [23]:
df_w_generated_traj_troll.traj_generated.values[0].shape
df_w_generated_traj_troll.traj_generated_reshaped.values[0].shape

(2, 50, 2)

In [ ]:
df_w_generated_traj_troll["generated_traj_policy"] = df_w_generated_traj_troll["policy"].values
df_w_generated_traj_organic["generated_traj_policy"] = df_w_generated_traj_organic["policy"].values

In [ ]:
values = [0.1,0.2,0.3,0.4,0.5,0.6]
chunk = 2475

df_w_generated_traj_troll["percentage_non_troll"] = [values[i] for i in (df_w_generated_traj_troll.index // chunk)]
df_w_generated_traj_organic["percentage_non_troll"] = [values[i] for i in (df_w_generated_traj_organic.index // chunk)]

In [ ]:
99.0*25

In [ ]:
num_runs = 10
n=100

running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

f1_list_rf = []
f1_list_xgb = []

misclassified_users_rf_all_subs = defaultdict(int)
misclassified_users_xgb_all_subs = defaultdict(int)

content_running_confusion_matrix_rf = np.zeros((2, 2), dtype=int)
content_running_confusion_matrix_xgb = np.zeros((2, 2), dtype=int)

content_f1_list_rf = []
content_f1_list_xgb = []

content_misclassified_users_rf_all_subs = defaultdict(int)
content_misclassified_users_xgb_all_subs = defaultdict(int)


for percentage_non_troll in [0.1,0.2,0.3,0.4,0.5,0.6]:
    for run in range(num_runs):
        
        seed = 100 + run
        print(percentage_non_troll,"run:",run)
        
        df_troll_sampled = df_w_generated_traj_troll[(df_w_generated_traj_troll.run == run) & (df_w_generated_traj_troll.percentage_non_troll == percentage_non_troll) & (df_w_generated_traj_troll.russian == 1)].copy()
        df_organic_sampled = df_w_generated_traj_organic[(df_w_generated_traj_organic.run == run) & (df_w_generated_traj_organic.percentage_non_troll == percentage_non_troll) & (df_w_generated_traj_organic.russian == 0)].copy()
            
        df_troll_sampled["feature_col"] = df_troll_sampled['generated_traj_policy'].apply(data_corruption_utils.normalize_replace_zeros)
        df_organic_sampled["feature_col"] = df_organic_sampled['generated_traj_policy'].apply(data_corruption_utils.normalize_replace_zeros)

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_troll_sampled,df_organic_sampled,seed = seed)

        print("policy overall_rf_f1",overall_rf_f1)
        f1_list_rf.append(overall_rf_f1)
        f1_list_xgb.append(overall_xgb_f1)
        running_confusion_matrix_rf += confusion_matrix_rf
        running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]
        
        df_troll_sampled["feature_col"] = df_troll_sampled['embed_generated'].apply(lambda x: x.flatten())
        df_organic_sampled["feature_col"] = df_organic_sampled['embed_generated'].apply(lambda x: x.flatten())

        confusion_matrix_rf, confusion_matrix_xgb, overall_rf_f1, overall_xgb_f1, misclassified_users_rf, misclassified_users_xgb = experiment_utils.run_experiment(df_troll_sampled,df_organic_sampled,seed = seed)

        print("content overall_rf_f1",overall_rf_f1)
        content_f1_list_rf.append(overall_rf_f1)
        content_f1_list_xgb.append(overall_xgb_f1)
        content_running_confusion_matrix_rf += confusion_matrix_rf
        content_running_confusion_matrix_xgb += confusion_matrix_xgb
        for user in misclassified_users_rf:
            content_misclassified_users_rf_all_subs[user] += misclassified_users_rf[user]
        for user in misclassified_users_xgb:
            content_misclassified_users_xgb_all_subs[user] += misclassified_users_xgb[user]


    print("dropped_percent", percentage_non_troll)
    print("Final Action Confusion Matrix (Random Forest):\n", running_confusion_matrix_rf)    
    f1 = np.mean(f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Action Confusion Matrix (XGBoost):\n", running_confusion_matrix_xgb)        
    f1 = np.mean(f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")

    print("dropped_percent", percentage_non_troll)
    print("Final Content Confusion Matrix (Random Forest):\n", content_running_confusion_matrix_rf)    
    f1 = np.mean(content_f1_list_rf)
    print(f"Mean F1 Score (Random Forest): {f1:.2f}\n")

    print("Final Content Confusion Matrix (XGBoost):\n", content_running_confusion_matrix_xgb)        
    f1 = np.mean(content_f1_list_xgb)
    print(f"Mean F1 Score (XGBoost): {f1:.2f}\n")


    results_mixed = {
        "f1_list_rf":f1_list_rf,
        "f1_list_xgb":f1_list_xgb,
        "running_confusion_matrix_rf":running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":misclassified_users_xgb_all_subs
    }

    results_mixed_content = {
        "f1_list_rf":content_f1_list_rf,
        "f1_list_xgb":content_f1_list_xgb,
        "running_confusion_matrix_rf":content_running_confusion_matrix_rf,
        "running_confusion_matrix_xgb":content_running_confusion_matrix_xgb,
        "misclassified_users_rf_all_subs":content_misclassified_users_rf_all_subs,
        "misclassified_users_xgb_all_subs":content_misclassified_users_xgb_all_subs
    }

    output_dir = project_output_path + "/detection_evade"
    os.makedirs(output_dir, exist_ok=True)
    with open(output_dir + "/results_mixed_policy_" + str(percentage_non_troll) +".pkl", "wb") as f:
        pickle.dump(results_mixed, f)
    with open(output_dir + "/results_mixed_content_" + str(percentage_non_troll) +".pkl", "wb") as f:
        pickle.dump(results_mixed_content, f)